# Myanmar Conflict Observatory: EDA
**Objective:** Exploratory Data Analysis of ACLED Myanmar conflict data (Post-Coup 2021 onwards).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import os
import glob

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

def get_latest_data():
    """
    Unified Data Ingestion Framework: 
    Automatically detects the most recent CSV file in the 'data/' directory.
    """
    data_dir = os.path.join(os.getcwd(), "data")
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        return "ERROR: 'data/' folder was missing (created). Please place your ACLED CSV there."
    
    files = glob.glob(os.path.join(data_dir, "*.csv"))
    if not files:
        return "ERROR: No CSV files found in 'data/' folder. Please place your ACLED dataset there."
    
    latest_file = max(files, key=os.path.getmtime)
    print(f"Using Dataset: {os.path.basename(latest_file)}")
    return latest_file

DATA_PATH = get_latest_data()

def load_and_clean_data(path):
    if "ERROR" in path:
        raise FileNotFoundError(path)
        
    df = pd.read_csv(path)
    # Filter for Myanmar only
    df = df[df['country'] == 'Myanmar']
    
    # Convert event_date to datetime
    df['event_date'] = pd.to_datetime(df['event_date'])
    
    # Sort by date
    df = df.sort_values('event_date')
    
    # Post-Coup filter (Feb 1, 2021 onwards)
    df = df[df['event_date'] >= '2021-02-01']
    
    print(f"Dataset Loaded: {len(df)} events found.")
    return df

df = load_and_clean_data(DATA_PATH)
df.head()

## 1. Temporal Analysis: Conflict Trends
Visualizing the frequency of events and fatalities over time.

In [ ]:
monthly_events = df.resample('ME', on='event_date').size().reset_index(name='event_count')
monthly_fatalities = df.resample('ME', on='event_date')['fatalities'].sum().reset_index()

plt.figure(figsize=(15, 6))
plt.plot(monthly_events['event_date'], monthly_events['event_count'], label='Number of Events', color='blue')
plt.plot(monthly_fatalities['event_date'], monthly_fatalities['fatalities'], label='Total Fatalities', color='red', linestyle='--')
plt.title("Myanmar Conflict Intensity: Events vs Fatalities (Post-Coup)", fontsize=15)
plt.legend()
plt.show()

## 2. Geospatial Analysis: State/Region Breakdown
Identifying hotspots where conflict is most concentrated.

In [ ]:
admin_stats = df.groupby('admin1').agg({
    'event_id_cnty': 'count',
    'fatalities': 'sum'
}).rename(columns={'event_id_cnty': 'event_count'}).sort_values('event_count', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x=admin_stats['event_count'], y=admin_stats.index, palette='viridis')
plt.title("Conflict Events by State/Region", fontsize=14)
plt.xlabel("Total Events")
plt.ylabel("Region")
plt.show()

## 3. Actor Analysis
Analyzing the primary actors involved in engagements.

In [ ]:
top_actors = pd.concat([df['actor1'], df['actor2']]).value_counts().head(10)

plt.figure(figsize=(10, 6))
top_actors.plot(kind='barh', color='darkred')
plt.title("Top 10 Most Active Actors")
plt.gca().invert_yaxis()
plt.show()

## 4. Sub-Event Type Breakdown
Understanding the nature of the conflict (Battles, Protests, Explosions).

In [ ]:
fig = px.pie(df, names='event_type', values='fatalities', title='Fatalities by Event Type')
fig.show()

## 5. Actor Normalization (Data Cleaning)
Categorizing messy ACLED actor names into clean groups for better analysis.

In [ ]:
def categorize_actor(actor_name):
    if pd.isna(actor_name): return "Unknown"
    name = str(actor_name).lower()
    
    if 'military forces of myanmar' in name or 'police forces of myanmar' in name:
        return 'SAC (Military/Police)'
    elif 'pdf' in name or 'people\'s defence force' in name or 'local defense force' in name:
        return 'PDF (Resistance)'
    elif 'protesters' in name:
        return 'Protesters'
    elif 'civilians' in name:
        return 'Civilians'
    # Common EAO acronyms
    elif any(eao in name for eao in ['knu', 'kia', 'tnla', 'mndaa', 'rcss', 'knpp', 'cnp', 'aa ']):
        return 'EAO (Ethnic Armed Orgs)'
    else:
        return 'Other/Militia'

df['actor1_clean'] = df['actor1'].apply(categorize_actor)
df['actor2_clean'] = df['actor2'].apply(categorize_actor)

# Visualize Cleaned Actor Interactions
actor_counts = df['actor1_clean'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=actor_counts.values, y=actor_counts.index, palette='magma')
plt.title("Simplified Actor Distribution")
plt.show()

## 6. Advanced Conflict Mapping
Improving the visualization with Time Animations and Intensity Heatmaps.

In [ ]:
# Prepare data for animation
df['year_month'] = df['event_date'].dt.strftime('%Y-%m')
df_animated = df.sort_values('event_date')

# 1. Animated Conflict Map (Evolution over time)
fig_anim = px.scatter_mapbox(
    df_animated, 
    lat="latitude", 
    lon="longitude", 
    color="actor1_clean",
    size="fatalities",
    animation_frame="year_month",
    hover_name="location",
    hover_data=["event_date", "event_type"],
    color_discrete_map={
        "SAC (Military/Police)": "#EF553B",
        "PDF (Resistance)": "#636EFA",
        "EAO (Ethnic Armed Orgs)": "#00CC96",
        "Civilians": "#AB63FA"
    },
    zoom=5, 
    height=700,
    title="Conflict Evolution in Myanmar (2021-2026)"
)
fig_anim.update_layout(mapbox_style="carto-darkmatter")
fig_anim.show()

# 2. Density Heatmap (Conflict Intensity)
fig_heat = px.density_mapbox(
    df, 
    lat='latitude', 
    lon='longitude', 
    z='fatalities', 
    radius=10,
    center=dict(lat=18.5, lon=96),
    zoom=5,
    mapbox_style="carto-darkmatter",
    height=700,
    title="Conflict Intensity Heatmap (Fatality Density)"
)
fig_heat.show()